# MyTravels Infrastructure Runbook

This notebook is the end-to-end operational runbook for standing up the full MyTravels Kubernetes infrastructure on a local k3d cluster. Run cells top to bottom to bring everything up from scratch.

## Summary

- **Step 1 — Prerequisites**: Verify the required tools are installed (Rancher Desktop, k3d, kubectl, JupyterLab, OpenLens).
- **Step 2 — Create the Cluster**: Create the local k3d cluster (1 control plane, 3 workers) with port mappings for Traefik HTTP (8080) and PostgreSQL TCP (5432).
- **Step 3 — /etc/hosts**: Add `*.mytravels.local` hostnames to `/etc/hosts` for host-based ingress routing.
- **Step 4 — Namespace**: Create the `mytravels-default` namespace that holds all resources.
- **Step 5 — Environment File**: Copy `.env.example` to `.env` and fill in real values.
- **Step 6 — PostgreSQL**: Deploy PostgreSQL 17.6 with secret, PVC, deployment, and ClusterIP service.
- **Step 7 — Database Migrations**: Run the once-off `db-migrations` Job (cleanup initContainer + the `efbundle` migrations executable).
- **Step 8 — RabbitMQ**: Deploy RabbitMQ with the management plugin enabled via ConfigMap.
- **Step 9 — MinIO**: Deploy MinIO object storage pinned to agent-2 with hostPath-backed PVs.
- **Step 10 — API**: Deploy the stateless ASP.NET Core REST API with its secret and service.
- **Step 11 — Messaging**: Deploy the stateless ASP.NET Core background worker that consumes RabbitMQ messages.
- **Step 12 — Traefik Configuration**: Add the `postgres` TCP entrypoint to Traefik via `HelmChartConfig`.
- **Step 13 — Ingress**: Apply the ingress rules exposing RabbitMQ, MinIO, the API, Messaging, and PostgreSQL through Traefik.
- **Step 14 — Backstage RBAC**: Create the read-only ServiceAccount, ClusterRole, and token for the Backstage Kubernetes plugin.
- **Step 15 — Full Stack Verification**: Confirm all pods, PVCs, services, ingresses, and URLs are healthy.
- **Step 16 — Diagnostics**: Pull logs and events per service when something misbehaves.
- **Step 17 — Teardown**: Delete all resources in reverse order and optionally the whole cluster.

## The application architecture  

![architecture](images/architecture.png)

---

## Step 1 — Prerequisites

Rancher Desktop, k3d, kubectl, JupyterLab, and Freelens/OpenLens install notes: [macOS](<../1-install tools (macos).md>) · [Ubuntu](<../1-install tools (ubuntu).md>) · [Windows](<../1-install tools (windows).md>).

Once Rancher Desktop is installed, open it and ensure the container engine is running before continuing.

In [ ]:
%%bash
echo "=== Docker ==="
docker --version
echo "=== k3d ==="
k3d --version
echo "=== kubectl ==="
kubectl version --client 2>/dev/null || kubectl version --client --short

---

## Step 2 — Create the Cluster

![cluster](images/k8s%20components.drawio.png)

Creates a local k3d cluster with 1 control plane node and 3 worker nodes. Traefik is bundled automatically by k3s and serves as the ingress controller.

| Flag | Meaning |
|---|---|
| `-p "8080:80@loadbalancer"` | Maps `localhost:8080` → cluster port 80 (Traefik web entrypoint) |
| `-p "5432:5432@loadbalancer"` | Maps `localhost:5432` → cluster port 5432 (Traefik postgres TCP entrypoint) |
| `--image ghcr.io/k3s-io/k3s:v1.35.3-k3s1` | Pins the k3s version for reproducible cluster creation |
| `--servers 1` | 1 control plane node |
| `--agents 3` | 3 worker nodes |

> Skip this cell if the cluster already exists (`k3d cluster list`).

> **Ensure Rancher Desktop is running before this cell.** k3d creates the cluster's nodes as Docker containers, so it needs a live Docker daemon — on Linux there's no system Docker install, Rancher Desktop *is* the daemon. If it isn't running (or hasn't finished starting its VM yet), `k3d cluster create` fails immediately with `Cannot connect to the Docker daemon at unix:///home/<user>/.rd/docker.sock`, because that socket file doesn't exist until Rancher Desktop creates it. Open Rancher Desktop and wait for it to fully start, then confirm with `docker info` before retrying.

In [ ]:
%%bash
k3d cluster create mytravels \
  -p "8080:80@loadbalancer" \
  -p "5432:5432@loadbalancer" \
  --image ghcr.io/k3s-io/k3s:v1.35.3-k3s1 \
  --servers 1 \
  --agents 3

In [ ]:
%%bash
echo "=== Nodes ==="
kubectl get nodes
echo ""
echo "=== Traefik ==="
kubectl get pods -n kube-system -l app.kubernetes.io/name=traefik

---

## Step 3 — /etc/hosts

The ingress rules in `9-ingress.yaml` use host-based routing. Add the entries below to your hosts file so your browser resolves the hostnames to localhost.

```
127.0.0.1  rabbitmq.mytravels.local
127.0.0.1  minio.mytravels.local
127.0.0.1  api.mytravels.local
127.0.0.1  messaging.mytravels.local
```

Run **one** of the next two cells depending on your OS:

- **macOS/Linux** — appends to `/etc/hosts` via `sudo`, prompting for your password.
- **Windows** — appends to `C:\Windows\System32\drivers\etc\hosts`. There's no `sudo` on Windows, so instead the cell checks whether it has Administrator privileges and writes directly if so. If not, close Jupyter/VS Code and relaunch it "as Administrator", then re-run the cell.

### macOS/Linux

In [ ]:
import subprocess
import getpass

password = getpass.getpass("sudo password: ")

hosts = ["rabbitmq.mytravels.local", "minio.mytravels.local", "api.mytravels.local", "messaging.mytravels.local"]

with open("/etc/hosts", "r") as f:
    current = f.read()

for host in hosts:
    if host in current:
        print(f"Already present: {host}")
    else:
        entry = f"127.0.0.1  {host}\n"
        result = subprocess.run(
            ["sudo", "-S", "tee", "-a", "/etc/hosts"],
            input=f"{password}\n{entry}",
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"Added: {host}")
        else:
            print(f"Failed: {host} — {result.stderr.strip()}")

### Windows

In [ ]:
import ctypes

hosts_path = r"C:\Windows\System32\drivers\etc\hosts"
hosts = ["rabbitmq.mytravels.local", "minio.mytravels.local", "api.mytravels.local", "messaging.mytravels.local"]

def is_admin():
    try:
        return bool(ctypes.windll.shell32.IsUserAnAdmin())
    except Exception:
        return False

if not is_admin():
    print("Not running as Administrator — the hosts file is not writable.")
    print("Close Jupyter/VS Code and relaunch it via 'Run as Administrator', then re-run this cell.")
else:
    with open(hosts_path, "r") as f:
        current = f.read()

    with open(hosts_path, "a") as f:
        for host in hosts:
            if host in current:
                print(f"Already present: {host}")
            else:
                f.write(f"127.0.0.1  {host}\n")
                print(f"Added: {host}")

---

## Step 4 — Namespace

All MyTravels resources live in the `mytravels-default` namespace. This must exist before any service manifests are applied.

In [ ]:
%%bash
kubectl apply -f manifests/1-namespace.yaml

In [ ]:
%%bash
kubectl get namespace mytravels-default

---

## Step 5 — Environment File

Copy `.env.example` to `.env` and fill in real values (API keys, tokens, credentials) before deploying. `.env` is gitignored — it's read by `docker-compose.yml` for local development. The Kubernetes secrets used in later steps are populated separately, directly in each `1-secret.yaml`.

> Skip this cell if `.env` already exists — it will not be overwritten.

In [ ]:
%%bash
if [ -f .env ]; then
  echo ".env already exists — skipping"
else
  cp .env.example .env
  echo "Copied .env.example to .env — edit it with real values before continuing"
fi

---

## Step 6 — PostgreSQL

Deploys PostgreSQL 17.6. The secret keys (`POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_DB`) match the docker-compose environment variable names exactly.

| File | Creates |
|---|---|
| `1-secret.yaml` | `postgres-secret` — DB credentials |
| `2-pvc.yaml` | `postgres-data-pvc` — 200Mi data volume |
| `3-deployment.yaml` | `postgres` deployment with liveness probe |
| `4-service.yaml` | `postgres` ClusterIP service on 5432 |

External access is via the Traefik `IngressRouteTCP` in `9-ingress.yaml`, or `kubectl port-forward svc/postgres 5432:5432 -n mytravels-default` for direct local access.

> **Before applying:** update `1-secret.yaml` with real base64-encoded values if needed:
> ```bash
> echo -n 'your-value' | base64
> ```

In [ ]:
%%bash
kubectl apply -f manifests/postgres

In [ ]:
%%bash
kubectl rollout status deployment/postgres -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l app=postgres

---

## Step 7 — Database Migrations

Runs two once-off tasks in sequence. The `cleanup-migrations` initContainer deletes specific rows from `EFMigrationsHistory`, then the `migrate-core-db` main container runs the `efbundle` migrations executable against `CoreDbContext`. The Job completes once — it is never restarted (`restartPolicy: Never`).

| docker-compose service | Kubernetes equivalent |
|---|---|
| `cleanup-migrations` | `initContainer: cleanup-migrations` in the `db-migrations` Job |
| `migrate-core-db` | main container in the `db-migrations` Job |
| `depends_on: postgres: service_healthy` | `pg_isready` loop inside `cleanup-migrations` |
| `restart: "no"` | `restartPolicy: Never` on the Job pod |

| File | Creates |
|---|---|
| `1-secret.yaml` | `migrations-secret` — `ConnectionStrings__CoreDbContext` |
| `2-job.yaml` | `db-migrations` Job |

Postgres credentials (`POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_DB`) are pulled from the existing `postgres-secret`.

> **Before applying:** populate `migrations/1-secret.yaml` with a real base64-encoded connection string:
> ```bash
> echo -n "Host=postgres;Port=5432;Database=CoreDb;Username=...;Password=..." | base64
> ```
> Replace `<base64-encoded-connection-string>` in `1-secret.yaml` with the output.

In [ ]:
%%bash
# kubectl delete job db-migrations -n mytravels-default --wait=true
kubectl apply -f manifests/migrations/

In [ ]:
%%bash
# Poll until the pod is scheduled (handles the case where the cell runs before the pod exists)
until kubectl get pod -l job-name=db-migrations -n mytravels-default 2>/dev/null | grep -q db-migrations; do
  echo "Waiting for pod to be scheduled..."; sleep 2
done

# Wait until the init container finishes (pod moves past PodInitializing)
kubectl wait pod -l job-name=db-migrations -n mytravels-default \
  --for=condition=Initialized --timeout=120s

# Wait for the job to complete — guarantees migrate-core-db has started and exited
kubectl wait job/db-migrations -n mytravels-default \
  --for=condition=Complete --timeout=300s

echo "--LIST JOBS--"
kubectl get job db-migrations -n mytravels-default
echo "--CLEANUP MIGRATION LOGS--"
kubectl logs -n mytravels-default -l job-name=db-migrations -c cleanup-migrations
echo "--MIGRATION LOGS--"
kubectl logs -n mytravels-default -l job-name=db-migrations -c migrate-core-db

---

## Step 8 — RabbitMQ

Deploys RabbitMQ 3 with the management plugin. The `5-configmap.yaml` must be applied before the deployment — it supplies the `enabled_plugins` file that activates the management UI. Without it, RabbitMQ starts with 0 plugins and the UI never comes up.

| File | Creates |
|---|---|
| `1-secret.yaml` | `rabbitmq-secret` — broker credentials |
| `2-pvc.yaml` | `rabbitmq-data-pvc` (200Mi) |
| `5-configmap.yaml` | `rabbitmq-config` — `enabled_plugins` file |
| `3-deployment.yaml` | `rabbitmq` deployment |
| `4-service.yaml` | `rabbitmq-amqp` (ClusterIP 5672) + `rabbitmq-management` (ClusterIP 15672) |

The management UI is accessible at [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) via the Traefik Ingress (Step 13).

In [ ]:
%%bash
kubectl apply -f manifests/rabbitmq/

In [ ]:
%%bash
kubectl rollout status deployment/rabbitmq -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l app=rabbitmq

In [ ]:
%%bash
# Confirm the management plugin loaded — should show 'completed with 3 plugins'
kubectl logs -n mytravels-default -l app=rabbitmq --tail=20 | grep -E 'plugin|completed'

---

## Step 9 — MinIO

Deploys MinIO (S3-compatible object storage). The deployment uses a `nodeSelector` pinned to `k3d-mytravels-agent-2` and manual PVs with `hostPath` mounts on that node.

| File | Creates |
|---|---|
| `1-secret.yaml` | `minio-secret` — root credentials |
| `2-pv-pvc.yaml` | PVs + PVCs for data (200Mi) and config (50Mi) |
| `3-deployment.yaml` | `minio` deployment pinned to agent-2 |
| `4-service.yaml` | `minio` (ClusterIP 9000) + `minio-console` (ClusterIP 9090) |

In [ ]:
%%bash
kubectl apply -f manifests/minio/

In [ ]:
%%bash
kubectl rollout status deployment/minio -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l app=minio

---

## Step 10 — API

Deploys the MyTravels ASP.NET Core REST API. The service is stateless — no PVC is required.

Secret keys match the docker-compose `environment` keys exactly. Non-sensitive config (`ASPNETCORE_ENVIRONMENT`, `ASPNETCORE_URLS`, public URLs, `MinIO__Endpoint`) is set directly in the Deployment rather than in the Secret.

| docker-compose env var | Kubernetes |
|---|---|
| `ConnectionStrings__CoreDbContext` | Secret `api-secret` → `secretKeyRef` |
| `GoogleApiKey` | Secret `api-secret` → `secretKeyRef` |
| `RabbitMQ__Uri` | Secret `api-secret` → `secretKeyRef` |
| `MinIO__AccessKey` | Secret `api-secret` → `secretKeyRef` |
| `MinIO__SecretKey` | Secret `api-secret` → `secretKeyRef` |
| `ASPNETCORE_ENVIRONMENT` | Direct env var `Production` |
| `ASPNETCORE_URLS` | Direct env var `http://+:5101` |
| `GoogleMapsUrl` | Direct env var `https://maps.googleapis.com` |
| `GooglePlacesUrl` | Direct env var `https://places.googleapis.com` |
| `MinIO__Endpoint` | Direct env var `minio:9000` (in-cluster DNS) |

| File | Creates |
|---|---|
| `1-secret.yaml` | `api-secret` — credentials and tokens |
| `2-deployment.yaml` | `api` deployment with liveness probe on `/health` |
| `3-service.yaml` | `api` ClusterIP service on 5101 |

The API is accessible at [http://api.mytravels.local:8080](http://api.mytravels.local:8080) via the Traefik Ingress (Step 13).

> **Before applying:** populate `api/1-secret.yaml` with real base64-encoded values:
> ```bash
> echo -n "your-value" | base64
> ```
>
> **Dependency:** RabbitMQ (Step 8) must be healthy and the migrations Job (Step 7) must have completed successfully before applying the API Deployment. The `depends_on` from docker-compose has no automatic equivalent in Kubernetes.

In [ ]:
%%bash
kubectl apply -f manifests/api/

In [ ]:
%%bash
kubectl rollout status deployment/api -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=api

---

## Step 11 — Messaging

Deploys the MyTravels ASP.NET Core background worker that consumes RabbitMQ messages. The service is stateless — no PVC is required.

Secret keys match the docker-compose `environment` keys exactly. Non-sensitive config (`ASPNETCORE_ENVIRONMENT`, `ASPNETCORE_URLS`, public URLs, `MinIO__Endpoint`) is set directly in the Deployment. Shared secrets (`ConnectionStrings__CoreDbContext`, `RabbitMQ__Uri`, `MinIO__AccessKey`, `MinIO__SecretKey`, `GoogleApiKey`) use the same base64 values as `api-secret` but are stored in the dedicated `messaging-secret`. `ContentSafetyEndpoint` and `ContentSafetyKey` are new keys not present in the API.

| docker-compose env var | Kubernetes |
|---|---|
| `ConnectionStrings__CoreDbContext` | Secret `messaging-secret` → `secretKeyRef` |
| `RabbitMQ__Uri` | Secret `messaging-secret` → `secretKeyRef` |
| `MinIO__AccessKey` | Secret `messaging-secret` → `secretKeyRef` |
| `MinIO__SecretKey` | Secret `messaging-secret` → `secretKeyRef` |
| `GoogleApiKey` | Secret `messaging-secret` → `secretKeyRef` |
| `ContentSafetyEndpoint` | Secret `messaging-secret` → `secretKeyRef` |
| `ContentSafetyKey` | Secret `messaging-secret` → `secretKeyRef` |
| `ASPNETCORE_ENVIRONMENT` | Direct env var `Production` |
| `ASPNETCORE_URLS` | Direct env var `http://+:5102` |
| `GoogleMapsUrl` | Direct env var `https://maps.googleapis.com` |
| `GooglePlacesUrl` | Direct env var `https://places.googleapis.com` |
| `MinIO__Endpoint` | Direct env var `minio:9000` (in-cluster DNS) |

| File | Creates |
|---|---|
| `1-secret.yaml` | `messaging-secret` — credentials and tokens |
| `2-deployment.yaml` | `messaging` deployment with liveness probe on `/health` |
| `3-service.yaml` | `messaging` ClusterIP service on 5102 |

The messaging worker has no user-facing UI, but it is exposed via Ingress at [http://messaging.mytravels.local:8080/health](http://messaging.mytravels.local:8080/health) for health checks (Step 13).

> **Before applying:** populate `messaging/1-secret.yaml` with real base64-encoded values for `ContentSafetyEndpoint` and `ContentSafetyKey`:
> ```bash
> echo -n "https://your-endpoint.cognitiveservices.azure.com/" | base64   # ContentSafetyEndpoint
> echo -n "your-key" | base64                                              # ContentSafetyKey
> ```
>
> **Dependency:** RabbitMQ (Step 8) must be healthy and MinIO (Step 9) must be running before the messaging worker can process messages.

In [ ]:
%%bash
kubectl apply -f manifests/messaging/

In [ ]:
%%bash
kubectl rollout status deployment/messaging -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=messaging

---

## Step 12 — Traefik Configuration

Adds the `postgres` TCP entrypoint to Traefik so that the `IngressRouteTCP` in `9-ingress.yaml` can route raw TCP connections on port 5432 to the postgres pod. Without this, connections to `127.0.0.1:5432` are refused even when postgres is healthy.

The `HelmChartConfig` patches the k3s-managed Traefik Helm release — k3s picks it up and restarts Traefik automatically within ~15 seconds.

| File | Creates |
|---|---|
| `8-traefik-config.yaml` | `HelmChartConfig/traefik` — adds `--entrypoints.postgres.address=:5432/tcp` |

> **Cluster port mapping:** the k3d cluster must have been created with `-p "5432:5432@loadbalancer"` (see Step 2) so that the Traefik entrypoint is reachable from the host.

In [ ]:
%%bash
kubectl apply -f manifests/8-traefik-config.yaml
echo ""
echo "Waiting for Traefik to restart..."
sleep 20
kubectl rollout status deploy/traefik -n kube-system --timeout=60s
echo ""
for i in $(seq 1 12); do
  ARGS=$(kubectl get deploy traefik -n kube-system -o jsonpath='{.spec.template.spec.containers[0].args}' | tr ',' '\n')
  if echo "$ARGS" | grep -q postgres; then
    echo "postgres entrypoint confirmed:"
    echo "$ARGS" | grep postgres
    break
  fi
  echo "Waiting for postgres entrypoint... ($i/12)"
  sleep 5
done

---

## Step 13 — Ingress

In [ ]:
%%bash
kubectl apply -f manifests/9-ingress.yaml

In [ ]:
%%bash
kubectl get ingress -n mytravels-default


Applies the top-level ingress rules that expose services through Traefik.

| Host | Routes to | Port | Protocol |
|---|---|---|---|
| [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) | `rabbitmq-management` service | 15672 | HTTP |
| [http://minio.mytravels.local:8080](http://minio.mytravels.local:8080) | `minio-console` service | 9090 | HTTP |
| [http://api.mytravels.local:8080](http://api.mytravels.local:8080/swagger) | `api` service | 5101 | HTTP |
| [http://messaging.mytravels.local:8080/health](http://messaging.mytravels.local:8080/health) | `messaging` service | 5102 | HTTP |
| `127.0.0.1:5432` (TCP) | `postgres` service | 5432 | TCP |

Traffic flow (HTTP): `localhost:8080` → k3d load balancer → Traefik (port 80) → service backend.
PostgreSQL: `localhost:5432` → k3d load balancer → Traefik (postgres entrypoint) → postgres service.

---

## Step 14 — Backstage RBAC

Creates the ServiceAccount and permissions that allow a local Backstage instance to read cluster state via the Kubernetes plugin.

| File | Creates |
|---|---|
| `10-backstage-rbac.yaml` | `backstage` ServiceAccount, `backstage-kubernetes-reader` ClusterRole + ClusterRoleBinding, `backstage-token` Secret |

The ClusterRole grants read-only access to pods, deployments, replicasets, services, ingresses, jobs, events, and namespaces — everything the Backstage Kubernetes tab needs.

> **After applying:** run the verify cell below to print the cluster URL, token, and CA data, then paste those values into the `kubernetes` section of `app-config.local.yaml` in your Backstage instance. The cluster URL changes each time the cluster is recreated — re-run this step and update the config whenever you rebuild.

In [ ]:
%%bash
kubectl apply -f manifests/10-backstage-rbac.yaml

In [ ]:
%%bash
# Confirm the ServiceAccount token was issued, then print the values needed
# for app-config.local.yaml in your Backstage instance
kubectl wait secret/backstage-token -n mytravels-default \
  --for=jsonpath='{.data.token}' --timeout=30s
echo ""
echo "=== ServiceAccount ==="
kubectl get serviceaccount backstage -n mytravels-default
echo ""
echo "=== Cluster URL ==="
kubectl config view --minify -o jsonpath='{.clusters[0].cluster.server}'
echo ""
echo ""
echo "=== Service Account Token ==="
kubectl get secret backstage-token -n mytravels-default \
  -o jsonpath='{.data.token}' | base64 -d
echo ""
echo ""
echo "=== CA Data (base64) ==="
kubectl config view --raw --minify \
  -o jsonpath='{.clusters[0].cluster.certificate-authority-data}'
echo ""

---

## Step 15 — Full Stack Verification

Run these cells to confirm all resources are healthy before using the stack.

In [ ]:
%%bash
echo "=== Pods ==="
kubectl get pods -n mytravels-default
echo ""
echo "=== PVCs ==="
kubectl get pvc -n mytravels-default
echo ""
echo "=== Services ==="
kubectl get svc -n mytravels-default
echo ""
echo "=== Ingress ==="
kubectl get ingress -n mytravels-default

In [ ]:
%%bash
echo "=== RabbitMQ Management ==="
curl -s -o /dev/null -w "%{http_code}" http://rabbitmq.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== MinIO Console ==="
curl -s -o /dev/null -w "%{http_code}" http://minio.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== API ==="
curl -s -o /dev/null -w "%{http_code}" http://api.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== Messaging ==="
curl -s -o /dev/null -w "%{http_code}" http://messaging.mytravels.local:8080/health && echo " OK" || echo " UNREACHABLE"
echo ""

In [ ]:
%%bash
kubectl top pods -A


**Services deployed:**

| Service | Purpose |
|---|---|
| PostgreSQL | Primary database |
| RabbitMQ | Message broker |
| MinIO | S3-compatible object storage |
| API | ASP.NET Core REST API |
| Messaging | ASP.NET Core background worker (RabbitMQ consumer) |

**Management UIs (after full setup):**

| Service | URL |
|---|---|
| RabbitMQ Management | [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) |
| MinIO Console | [http://minio.mytravels.local:8080](http://minio.mytravels.local:8080) |
| API | [http://api.mytravels.local:8080/swagger](http://api.mytravels.local:8080/swagger) |


---

## Step 16 — Diagnostics

Run these cells when a service is not behaving as expected.

In [ ]:
%%bash
echo "=== PostgreSQL logs ==="
kubectl logs -n mytravels-default -l app=postgres --tail=20

In [ ]:
%%bash
echo "=== RabbitMQ logs ==="
kubectl logs -n mytravels-default -l app=rabbitmq --tail=20

In [ ]:
%%bash
echo "=== MinIO logs ==="
kubectl logs -n mytravels-default -l app=minio --tail=20

In [ ]:
%%bash
echo "=== API logs ==="
kubectl logs -n mytravels-default -l app=api --tail=20

In [ ]:
%%bash
echo "=== Messaging logs ==="
kubectl logs -n mytravels-default -l app=messaging --tail=20

In [ ]:
%%bash
# Recent events — useful for diagnosing scheduling or PVC binding failures
kubectl get events -n mytravels-default --sort-by='.lastTimestamp' | tail -20

---

## Step 17 — Teardown

Delete all resources and the cluster. Run cells individually to tear down selectively, or run all to wipe everything.

In [ ]:
%%bash
# Delete all manifests in reverse order
kubectl delete -f manifests/9-ingress.yaml
kubectl delete -f manifests/8-traefik-config.yaml
kubectl delete -f manifests/10-backstage-rbac.yaml
kubectl delete -f manifests/messaging/
kubectl delete -f manifests/api/
kubectl delete -f manifests/minio/
kubectl delete -f manifests/rabbitmq/
kubectl delete -f manifests/migrations/
kubectl delete -f manifests/postgres/
kubectl delete -f manifests/1-namespace.yaml

In [ ]:
%%bash
# Delete the entire cluster — removes all Docker containers and volumes
k3d cluster delete mytravels